In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import pickle

In [ ]:
load_dotenv()

server = os.getenv('SQL_SERVER')
database = os.getenv('SQL_DATABASE')

connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        print(f"Conectado com sucesso ao banco: {database}")
except Exception as e:
    print(f"Erro na conexão: {e}")

In [ ]:
# Carregar clientes em revisão
query = "SELECT * FROM decisoes_credito WHERE decisao = 'REVISAO'"
df_revisao = pd.read_sql(query, engine)

print(f"Clientes em revisão: {len(df_revisao)}")
print(df_revisao.head())

In [ ]:
# Normalizar variáveis para escala 0 a 1
df_revisao['score_normalizado'] = df_revisao['score_credito'] / 1000
df_revisao['renda_normalizada'] = (df_revisao['renda_mensal'] - df_revisao['renda_mensal'].min()) / (df_revisao['renda_mensal'].max() - df_revisao['renda_mensal'].min())
df_revisao['valor_normalizado'] = (df_revisao['valor_solicitado'] - df_revisao['valor_solicitado'].min()) / (df_revisao['valor_solicitado'].max() - df_revisao['valor_solicitado'].min())

# Inverter o risco (menor risco = maior prioridade)
df_revisao['risco_numerico'] = df_revisao['probabilidade_risco'].str.replace('%', '').astype(float) / 100
df_revisao['risco_invertido'] = 1 - df_revisao['risco_numerico']

# Calcular pontuação de prioridade
df_revisao['pontuacao'] = (
    (df_revisao['score_normalizado'] * 0.3) +
    (df_revisao['renda_normalizada'] * 0.3) +
    (df_revisao['valor_normalizado'] * 0.2) +
    (df_revisao['risco_invertido'] * 0.2)
)

print("Pontuação calculada com sucesso!")
print(df_revisao[['nome', 'score_credito', 'renda_mensal', 'pontuacao']].sort_values('pontuacao', ascending=False).head(10))

In [ ]:
# Simular horário de entrada na fila
agora = datetime.now()
df_revisao['entrada_fila'] = [
    agora - timedelta(hours=random.uniform(0, 24)) 
    for _ in range(len(df_revisao))
]

# Calcular tempo de espera em horas
df_revisao['horas_espera'] = (agora - df_revisao['entrada_fila']).dt.total_seconds() / 3600

# Definir semáforo combinando tempo + pontuação
def definir_semaforo(row):
    if row['horas_espera'] > 20 or row['pontuacao'] > 0.7:
        return 'VERMELHO'
    elif row['horas_espera'] > 8 or row['pontuacao'] > 0.4:
        return 'AMARELO'
    else:
        return 'VERDE'

df_revisao['semaforo'] = df_revisao.apply(definir_semaforo, axis=1)

print("Semáforo definido com sucesso!")
print(df_revisao['semaforo'].value_counts())

In [ ]:
# Selecionar colunas relevantes para o analista
colunas_fila = [
    'nome', 'cpf', 'idade', 'renda_mensal', 'score_credito',
    'valor_solicitado', 'prazo_meses', 'probabilidade_risco',
    'motivo', 'pontuacao', 'semaforo', 'entrada_fila', 'horas_espera'
]

df_fila = df_revisao[colunas_fila].sort_values('pontuacao', ascending=False)

# Salvar no banco
try:
    df_fila.to_sql('fila_revisao', engine, if_exists='replace', index=False)
    print(f"Fila salva com sucesso!")
    print(f"Total de clientes na fila: {len(df_fila)}")
except Exception as e:
    print(f"Erro ao salvar: {e}")

In [ ]:
# Visualizar fila final
print("=== FILA DE REVISÃO PRIORIZADA ===\n")

print("🔴 URGENTE:")
df_vermelho = df_fila[df_fila['semaforo'] == 'VERMELHO'][['nome', 'renda_mensal', 'score_credito', 'valor_solicitado', 'probabilidade_risco', 'horas_espera']].head(5)
print(df_vermelho.to_string())

print("\n🟡 ATENÇÃO:")
df_amarelo = df_fila[df_fila['semaforo'] == 'AMARELO'][['nome', 'renda_mensal', 'score_credito', 'valor_solicitado', 'probabilidade_risco', 'horas_espera']].head(5)
print(df_amarelo.to_string())

print("\n🟢 NORMAL:")
df_verde = df_fila[df_fila['semaforo'] == 'VERDE'][['nome', 'renda_mensal', 'score_credito', 'valor_solicitado', 'probabilidade_risco', 'horas_espera']].head(5)
print(df_verde.to_string())